# Week 6: referral specialty model selection
Run from a fresh kernel. The source CSV is loaded from `data/mtsamples (1).csv`, or downloaded from the public repository if absent. The code saves train, validation, and frozen test partitions under `week6_results/`. Do not tune using the frozen test set.
**Scope:** Nine proxy specialty labels from MTSamples; this is not clinical referral validation. The keyword rule is the PM Plan baseline. Primary metric: top-1 agreement; macro F1 is secondary.


In [1]:
"""Reproducible Week 6 experiment. Run with: python week6_workflow.py"""
from pathlib import Path
import hashlib, json, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

ROOT = Path.cwd()
SOURCE = ROOT / 'data' / 'mtsamples (1).csv'
if not SOURCE.exists():
    import urllib.request
    SOURCE.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Sade421/Zoticus-AI/main/mtsamples-v3.csv', SOURCE)
OUT = ROOT / 'week6_results'
OUT.mkdir(exist_ok=True)
SEED = 42
LABELS = ['Orthopedic', 'Gastroenterology', 'Neurology', 'Obstetrics / Gynecology',
          'Urology', 'ENT - Otolaryngology', 'Ophthalmology', 'Nephrology', 'Dermatology']
STOP = re.compile(r'(?i)(?:^|[\n,;.]\s*)(?:assessment(?:\s+and\s+plan)?|impression|(?:preoperative|postoperative|post-operative|pre-operative|final|discharge|admission)\s+diagnos(?:is|es)|diagnos(?:is|es)|plan|procedure(?:s|\s+performed)?|operation|operative\s+(?:report|procedure)|recommendations?|disposition|consult(?:ation)?\s+(?:requested|recommended))\s*[:\-]')
SPECIALTY = re.compile(r'(?i)\b(?:orthop(?:edic|aed)ics?|gastroenterolog(?:y|ist)|neurolog(?:y|ist)|(?:obstetric(?:s|ian)?|gynecolog(?:y|ist))|urolog(?:y|ist)|otolaryngolog(?:y|ist)|ophthalmolog(?:y|ist)|nephrolog(?:y|ist)|dermatolog(?:y|ist))\b')

def prepare(raw):
    d = raw.dropna(subset=['transcription', 'medical_specialty']).copy()
    d['medical_specialty'] = d.medical_specialty.astype(str).str.strip()
    d['raw_text'] = d.transcription.astype(str).str.strip()
    d = d[d.raw_text.ne('') & d.medical_specialty.isin(LABELS)].copy()
    # A text associated with two supported outcomes has no unique class target.
    conflicting = d.groupby('raw_text').medical_specialty.nunique()
    d = d[~d.raw_text.isin(conflicting[conflicting.gt(1)].index)].copy()
    d = d.drop_duplicates('raw_text').copy()
    d['text'] = d.raw_text.map(lambda s: SPECIALTY.sub('[specialty removed]', s[:STOP.search(s).start()] if STOP.search(s) else s))
    d['text'] = d.text.str.replace(r'\s+', ' ', regex=True).str.strip()
    # Reject notes where a heading survived punctuation/format variation.
    d = d[~d.text.str.contains(r'(?i)\b(?:preoperative|postoperative|post-operative|pre-operative|final)\s+diagnos(?:is|es)\b|\b(?:procedure performed|operative report)\b', regex=True)].copy()
    d = d[d.text.str.len().ge(40)].copy()
    # Identical model inputs cannot cross partitions, even if raw notes differ.
    conflicting = d.groupby('text').medical_specialty.nunique()
    d = d[~d.text.isin(conflicting[conflicting.gt(1)].index)].drop_duplicates('text').copy()
    return d

def keyword_rule(text, majority):
    rules = [('Urology', r'\b(?:urinar|bladder|prostate|hematuria|kidney stone)\w*'),
             ('Gastroenterology', r'\b(?:abdom|stomach|colon|diarrhea|esophag|reflux)\w*'),
             ('Neurology', r'\b(?:seizure|headache|migraine|stroke|numbness|weakness)\w*'),
             ('Orthopedic', r'\b(?:fracture|joint|knee|shoulder|hip|bone)\w*'),
             ('Obstetrics / Gynecology', r'\b(?:pregnan|uterus|vaginal|menstrual|ovarian)\w*'),
             ('ENT - Otolaryngology', r'\b(?:ear|sinus|tonsil|nasal|hearing)\w*'),
             ('Ophthalmology', r'\b(?:eye|vision|retina|cornea|cataract)\w*'),
             ('Nephrology', r'\b(?:renal|dialysis|creatinine|glomerul)\w*'),
             ('Dermatology', r'\b(?:skin|rash|lesion|eczema|psoriasis)\w*')]
    scores = {label: len(re.findall(pattern, text, flags=re.I)) for label, pattern in rules}
    best = max(scores.values())
    return majority if best == 0 else next(label for label, _ in rules if scores[label] == best)

def measure(name, y, pred):
    return {'model': name, 'validation_n': len(y), 'top1_agreement': round(accuracy_score(y, pred), 4),
            'macro_f1': round(f1_score(y, pred, labels=LABELS, average='macro', zero_division=0), 4)}

def main():
    raw = pd.read_csv(SOURCE)
    data = prepare(raw)
    train, other = train_test_split(data, test_size=.30, stratify=data.medical_specialty, random_state=SEED)
    val, test = train_test_split(other, test_size=.50, stratify=other.medical_specialty, random_state=SEED)
    assert not (set(train.text) & set(val.text) or set(train.text) & set(test.text) or set(val.text) & set(test.text))
    # The test labels are saved for later final evaluation, never used in this selection run.
    for name, part in [('train', train), ('validation', val), ('test_frozen', test)]:
        part[['raw_text', 'text', 'medical_specialty']].to_csv(OUT / f'{name}.csv', index=False)
    majority = train.medical_specialty.mode().iloc[0]
    y = val.medical_specialty
    rows = [measure('Majority class', y, [majority] * len(val)),
            measure('Keyword rule', y, [keyword_rule(t, majority) for t in val.text])]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=4000,
                                 sublinear_tf=True, stop_words='english')
    X_train = vectorizer.fit_transform(train.text).toarray().astype('float32')
    X_val = vectorizer.transform(val.text).toarray().astype('float32')
    models = [('MLP 32', (32,)), ('MLP 64-32', (64, 32))]
    preds = {}
    for name, shape in models:
        clf = MLPClassifier(hidden_layer_sizes=shape, alpha=.01, early_stopping=True,
                            validation_fraction=.15, n_iter_no_change=5, max_iter=35,
                            random_state=SEED)
        clf.fit(X_train, pd.Categorical(train.medical_specialty, categories=LABELS).codes)
        p = np.array(LABELS)[clf.predict(X_val)]
        preds[name] = p
        rows.append(measure(name, y, p))
    winner = max(rows[2:], key=lambda r: (r['top1_agreement'], r['macro_f1']))['model']
    clf = dict(models)[winner]
    # Local occlusion: remove each of the 15 most salient nonzero tokens for one validation case.
    fitted = None
    # Refit the selected configuration with the same deterministic parameters for explanation.
    fitted = MLPClassifier(hidden_layer_sizes=clf, alpha=.01, early_stopping=True,
                           validation_fraction=.15, n_iter_no_change=5, max_iter=35,
                           random_state=SEED).fit(X_train, pd.Categorical(train.medical_specialty, categories=LABELS).codes)
    idx = 0
    x = X_val[idx].copy()
    target = LABELS[fitted.predict([x])[0]]
    class_idx = list(fitted.classes_).index(LABELS.index(target))
    probability = fitted.predict_proba([x])[0, class_idx]
    candidates = np.flatnonzero(x)
    candidates = candidates[np.argsort(x[candidates])[-15:]]
    effects = []
    for j in candidates:
        altered = x.copy(); altered[j] = 0
        effects.append({'token': vectorizer.get_feature_names_out()[j],
                        'probability_drop': round(float(probability - fitted.predict_proba([altered])[0, class_idx]), 4)})
    effects.sort(key=lambda v: v['probability_drop'], reverse=True)
    report = {'source_shape': list(raw.shape), 'prepared_n': len(data),
              'class_counts': data.medical_specialty.value_counts().to_dict(),
              'train_n': len(train), 'validation_n': len(val), 'frozen_test_n': len(test),
              'source_sha256': hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
              'results': rows, 'selected_by_validation': winner,
              'validation_recall': dict(zip(LABELS, [round(float(v), 4) for v in recall_score(y, preds[winner], labels=LABELS, average=None, zero_division=0)])),
              'confusion_matrix': confusion_matrix(y, preds[winner], labels=LABELS).tolist(),
              'interpretability_example': {'prediction': target, 'true_label': y.iloc[idx], 'token_occlusion': effects}}
    (OUT / 'results.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))

if __name__ == '__main__': main()


{
  "source_shape": [
    4999,
    6
  ],
  "prepared_n": 573,
  "class_counts": {
    "Neurology": 157,
    "Orthopedic": 101,
    "Gastroenterology": 85,
    "Obstetrics / Gynecology": 59,
    "Urology": 53,
    "Ophthalmology": 35,
    "Nephrology": 33,
    "ENT - Otolaryngology": 33,
    "Dermatology": 17
  },
  "train_n": 401,
  "validation_n": 86,
  "frozen_test_n": 86,
  "source_sha256": "83a65fd7ca071930264e27e6fb011bc5303a21294ff734d7a8198cfbb1db7a2a",
  "results": [
    {
      "model": "Majority class",
      "validation_n": 86,
      "top1_agreement": 0.2791,
      "macro_f1": 0.0485
    },
    {
      "model": "Keyword rule",
      "validation_n": 86,
      "top1_agreement": 0.7326,
      "macro_f1": 0.6994
    },
    {
      "model": "MLP 32",
      "validation_n": 86,
      "top1_agreement": 0.6163,
      "macro_f1": 0.4284
    },
    {
      "model": "MLP 64-32",
      "validation_n": 86,
      "top1_agreement": 0.314,
      "macro_f1": 0.0928
    }
  ],
  "selected_by

## Interpretation and next action
Prepared 573 cases; train 401, validation 86, frozen test 86. The keyword baseline scored 73.3% and the best neural candidate scored 61.6% on validation. The neural candidate does not beat the committed keyword baseline. Keep the keyword rule as the workflow comparator; improve and re-evaluate the neural approach before selecting it for Milestone 2.
